# Notebook zur erstellung des Datasets

In [57]:
import os
import random
import json
import pandas as pd

### Parameter

In [58]:
DATA_DIR = "../sensor_and_video_data"
LABELED_DIR = "../Labeled"
SEED = 42
# Files expeted in a run folder for it to be considered complete
EXPECTED_FILES = ["Timestamps", "synch_data"]


# Split ratios
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15


### Helper functions

In [59]:
def check_run_complete(folder, keywords):
    files = os.listdir(os.path.join(DATA_DIR, folder))

    for word in keywords:
        if not any(word in file for file in files):
            return False
    return True

### Runweise Aufteilung in Train/Validation/Test + auschließen von unvollständigen Runs

In [60]:

# Collect all run directories
runs = [
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
]

# Exclude incomplete runs
for run in runs[:]:
    if not check_run_complete(run, EXPECTED_FILES):
        runs.remove(run)
        print(f"Removed incomplete run: {run}")

print(f"Found {len(runs)} runs.")

# Shuffle runs for randomness
random.seed(SEED)
random.shuffle(runs)

# Split into train, val, test
num_runs = len(runs)
train_split = int(TRAIN_SPLIT * num_runs)
val_split = int((TRAIN_SPLIT + VAL_SPLIT) * num_runs)
train_runs = runs[:train_split]
val_runs = runs[train_split:val_split]
test_runs = runs[val_split:]

print(train_runs)
print(val_runs)
print(test_runs)

Found 16 runs.
['0721_0853_noheadsensor', '0724_0723', '0720_0828', '0720_0858', '0802_0800', '0725_0709', '0727_0812', '0721_0928_noheadsensor', '0713_0903', '0713_0932', '0801_0758']
['0803_0734', '0720_0748']
['0727_0735', '0713_0833', '0717_0717']


### Kombination von Head, Wrist und Seat Datensätzen

In [ ]:
def create_label_vector(run_file, len_of_run):
    file_path = os.path.join(LABELED_DIR, run_file)
    if os.path.isfile(file_path):
        print(f"Label file exists for {run_file}")
        label_vector = ["nothing"] * len_of_run

        with open(file_path, 'r') as f:
            data = json.load(f)

        raw = data["button_presses"]
        sequence_width_half = int(data["sequence_width"]) // 2

        button_presses = []
        for entry in raw.split(";"):
            entry = entry.strip()
            if not entry:
                continue
            label, idx = entry.split(":")
            button_presses.append((int(float(idx.strip()) * 2.4), label.strip())) # convert from video frames to sensor samples

        for idx, label in button_presses:
            if 0 <= idx < len_of_run:
                label_vector[idx-sequence_width_half : idx + sequence_width_half + 1] = [label] * (2 * sequence_width_half + 1)

        one_hot_labels = pd.get_dummies(label_vector)
        return one_hot_labels
        
    else:
        print(f"Label file missing for {run_file}")


create_label_vector("0717_0717_sequences.json", 10000)
    

Label file exists for 0717_0717_sequences.json
      Jumping   Left  Right  nothing
0       False  False  False     True
1       False  False  False     True
2       False  False  False     True
3       False  False  False     True
4       False  False  False     True
...       ...    ...    ...      ...
9995    False  False  False     True
9996    False  False  False     True
9997    False  False  False     True
9998    False  False  False     True
9999    False  False  False     True

[10000 rows x 4 columns]
